# download

> Download `ltuncay/Audio-JEPA` and write a numpy checkpoint.
>
> The Hugging Face repo ships a Lightning `.ckpt`, so conversion needs
> PyTorch. It keeps the context-encoder tensors and writes the file the
> runtime loads.


In [ ]:
#| default_exp download


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
from pathlib import Path

import numpy as np

from nada.core import repo_root
from nada.weights import encoder_key, usable


In [ ]:
#| export
REPO = "ltuncay/Audio-JEPA"


In [ ]:
#| export
def download(root: Path | None = None) -> Path:
    """Fetch `JEPA.ckpt`, write `models/audio_jepa.safetensors`, and return that path."""
    from huggingface_hub import hf_hub_download

    destination = (root or repo_root()) / "models" / "audio-jepa"
    destination.mkdir(parents=True, exist_ok=True)
    source = Path(hf_hub_download(REPO, "JEPA.ckpt", local_dir=destination))
    output = (root or repo_root()) / "models" / "audio_jepa.safetensors"
    convert(source, output)
    return output


In [ ]:
#| export
def convert(source: Path, output: Path) -> None:
    """Keep `encoder.*` tensors and write float32 safetensors. Needs PyTorch."""
    import torch
    from safetensors.numpy import save_file

    state = torch.load(source, map_location="cpu", weights_only=False)
    raw = state.get("state_dict", state)
    arrays = {}
    for key, value in raw.items():
        if not key.startswith("encoder.") or not usable(key):
            continue
        arrays[encoder_key(key)] = np_contiguous(value.detach().cpu().numpy())
    output.parent.mkdir(parents=True, exist_ok=True)
    save_file(arrays, output)
    print(f"wrote {output} ({len(arrays)} tensors)")


In [ ]:
#| export
def np_contiguous(array):
    if np.issubdtype(array.dtype, np.floating):
        array = array.astype(np.float32, copy=False)
    return np.ascontiguousarray(array)


In [ ]:
#| export
#| eval: false
if __name__ == "__main__":
    download()
